In [ ]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
import vbn_utils

In [ ]:
#Paths to all of the useful supplemental tables and tensors
sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
sessions_table = pd.read_csv(sessions_table_file)
good_sessions = sessions_table[sessions_table['abnormal_activity'].isnull() & sessions_table['abnormal_histology'].isnull()]['ecephys_session_id'].values

## Decoding through the omission

### Generated on HPC by 'run_image_decoding_sliding_window.py'

In [ ]:
all_session = []
cluster = 'all'
unitsamplesize =80
for session_id in good_sessions:
    try:
        sess_data = np.load(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/image_decoding_sliding_window", str(session_id) + f"_throughomissionRS_{cluster}.npy"), allow_pickle=True).item()
        if len(sess_data['VISall'][unitsamplesize]['imagewise_recall'])>0:
            all_session.append(np.array(sess_data['VISall'][unitsamplesize]['imagewise_recall']).mean(axis=1))
    except Exception as e:
        print(f"Error processing session {session_id}: {e}")

In [ ]:
from matplotlib.patches import Rectangle
plt.rcParams['figure.dpi'] = 300

fig, ax = plt.subplots()
fig.suptitle('VIS all, pop size: 80', fontsize=12)
fig.set_size_inches([8, 4])
vbn_utils.mean_sem_plot(np.array(all_session), ax,  x=sess_data['decodeWindows'], color='k')
ax.axhline(0.125, color='k', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time from pre-omission image onset (ms)')
ax.set_ylabel('Decoder accuracy')

filled_rect = Rectangle((0, ax.get_ylim()[1]), 250, 0.05, color='gray', alpha=1, clip_on=False)
ax.add_patch(filled_rect)

open_rect = Rectangle((750, ax.get_ylim()[1]), 250, 0.05, edgecolor='gray', facecolor='none', linewidth=2, clip_on=False)
ax.add_patch(open_rect)


vbn_utils.formatFigure(fig, ax, fontsize=16)